In [37]:
import os, json, glob, pandas as pd
import plotly.express as px

In [38]:
files = glob.glob(os.path.join('sweep_results', '*.json'))
jsons = [json.load(open(p)) for p in files]

df = pd.DataFrame(columns=['dtype', 'grd_ckpt', 'img_size'] + list(jsons[0]['results'][0].keys()))
for file, j in zip(files, jsons):
    results = j['results']
    for i in range(len(results)):
        
        name, _ = os.path.splitext(os.path.basename(file))
        tokens = name.split('_')
        
        results[i]['dtype'] = tokens[0]
        results[i]['grd_ckpt'] = True if tokens[1] == 'ckpt-on' else False
        results[i]['img_size'] = tokens[2].replace('img', '')
        
        df.loc[len(df)] = results[i]
        
df = df.sort_values(by=['dtype', 'grd_ckpt', 'img_size', 'batch_size']).reset_index(drop=True)

In [39]:
df

,dtype,grd_ckpt,img_size,batch_size,oom,fwd_ms,bwd_ms,opt_ms,step_ms,step_ms_std,prep_ms_cpu,samples_per_s,tokens_per_s,mean_seq_len,peak_allocated_gib,peak_reserved_gib,after_fwd_gib
0,bfloat16,False,336,1,False,35.227053,50.617373,47.137865,132.982291,2.743995,3.658025,7.519798,2573.274970,342.2,17.055762,18.087891,13.514434
1,bfloat16,False,336,2,False,42.039658,69.170513,47.153635,158.363806,9.445255,7.849024,12.629148,4504.817214,356.7,17.091330,19.142578,15.295969
2,bfloat16,False,336,4,False,67.523373,121.621913,47.169412,236.314698,30.784068,23.624279,16.926581,6494.729318,383.7,24.554636,33.822266,19.008916
3,bfloat16,False,336,8,False,145.742458,256.052582,47.162205,448.957245,95.052922,21.278029,17.819069,7549.939418,423.7,37.238005,42.761719,27.259671
4,bfloat16,False,336,16,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,bfloat16,False,448,1,False,34.467385,51.282550,47.094528,132.844463,3.147743,4.060699,7.527600,3298.594388,438.2,17.068207,18.175781,14.200457
6,bfloat16,False,448,2,False,48.733383,86.624368,47.101744,182.459495,8.853766,9.253339,10.961337,4962.197232,452.7,18.572950,23.636719,16.699478
7,bfloat16,False,448,4,False,85.162969,162.997940,47.213184,295.374093,31.704796,13.595127,13.542149,6496.168905,479.7,27.802501,38.755859,21.815119
8,bfloat16,False,448,8,True,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,bfloat16,False,672,1,False,42.709613,75.814637,47.162377,165.686627,4.506885,5.990877,6.035490,4123.446853,683.2,17.209094,19.203125,15.976238


In [40]:
for dtype in df['dtype'].unique():

    subset = df[df['dtype'] == dtype]

    fig = px.scatter(
        subset.dropna(),
        x='batch_size',
        y='samples_per_s',
        color='img_size',
        symbol='grd_ckpt',
        width=700,
        height=500,
        title=f'Throughput vs. batch size ({dtype})',
        size_max=100
    )
    fig.update_traces(marker=dict(size=15))
    fig.show()

In [41]:
for dtype in df['dtype'].unique():

    subset = df[df['dtype'] == dtype]

    fig = px.scatter(
        subset.dropna(),
        x='batch_size',
        y='peak_reserved_gib',
        color='img_size',
        symbol='grd_ckpt',
        width=700,
        height=500,
        title=f'VRAM usage vs. batch size ({dtype})',
        size_max=100
    )
    fig.update_traces(marker=dict(size=15))
    fig.show()